# Day 23: Bridge to LLMs

Today we look **inside ChatGPT-style models**. We build a tiny one ourselves, **from the bottom floor to the top floor**.

We use **one sentence** all day:

> **"i drink sweet ___"** → the model must guess **tea**

## 📋 Agenda

**Today we climb 8 floors (layers). Each floor fixes a problem of the floor below.**

```
TOP     Layer 7   The real LLM          the same recipe, but huge
          ▲
        Layer 6   Repeat               write a full sentence, word by word
          ▲
        Layer 5   Softmax              scores → chances → pick the next word
          ▲
        Layer 4   Attention            look at the useful words        ← the heart of the Transformer
          ▲
        Layer 3   Words → numbers      tokens, IDs, embeddings
          ▲
        Layer 2   Count (more words)   works, but breaks on new phrases
          ▲
        Layer 1   Count and guess      works, but gets a tie
          ▲
BOTTOM  Layer 0   The goal              guess the next word
```

**By the end you can explain:**
- What an **LLM** and an **SLM** are, and what **parameters** are
- What a **token**, an **ID** and an **embedding** are
- What **attention** does, and why a **Transformer** needs it
- How an LLM writes a long answer: **guess, add, repeat**

**Our plan:** at every layer we (1) learn the new words, (2) build one small piece, (3) see what **breaks**, and (4) climb to the next layer to fix it.

🎁 **Bonus (optional, at the end):** the same example with a real embedding model.


## First, the big words

| Word | Meaning |
|---|---|
| **Language** | Human words (English, Urdu) and code. |
| **Model** | A program that learned patterns from examples. Your Day 19–22 networks were models too. |
| **LLM** | **L**arge **L**anguage **M**odel. A very big model that guesses the next word. Examples: ChatGPT, Claude, Gemini. |
| **Parameters** (weights) | The numbers a model **learns** during training. "Large" means a huge number of them. |
| **SLM** | **S**mall **L**anguage **M**odel. The same idea, but smaller. It runs on a laptop or phone. |
| **Transformer** | The **design** (shape) of the network inside an LLM, like a CNN is the design for images. |

**How big?**

| Model | Parameters |
|---|---|
| Our Day 19 network | 147 |
| Our Day 20 network | 5,130 |
| SLM | 1 to 8 billion |
| LLM | 100 billion or more |

*Everyday example:* an LLM is a big hospital with every kind of doctor. An SLM is a village clinic: fast and cheap, but not for everything.

## The map

![Transformer, annotated](Visual%20Explanation/Day_23_Transformer_Annotated.png)

Read the picture from the **bottom to the top**. We build one box at a time. At the end we come back here and it will make sense.

| Layer | Idea | Problem it solves |
|---|---|---|
| 0 | The goal: autocomplete | (the goal) |
| 1 | Count and guess | – |
| 2 | Count with more words | Layer 1 gets a **tie** |
| 3 | Words become **numbers** | Layer 2 only knows phrases it has **seen** |
| 4 | **Attention** | A tie again, now with numbers |
| 5 | **Softmax** | Scores are not chances yet |
| 6 | **Repeat** | We only guessed one word |
| 7 | The real LLM | Our toy is tiny |

In [35]:
import numpy as np                      # vectors and tables of numbers
from collections import Counter         # counts how many times each item appears
np.set_printoptions(precision=3, suppress=True)   # print 3 decimals, no e-notation


def softmax(scores):                    # same softmax as Day 21
    e = np.exp(scores)                  # 1. e to the power of each score
    return e / e.sum()                  # 2. divide by the total → chances that add up to 1

---
# Class Problem: "i drink sweet ___"

## Layer 0 (bottom): The goal is autocomplete

🪜 **You are here: Layer 0 of 7**

| New word | Meaning |
|---|---|
| **Next-word prediction** | Look at the text so far, and guess the word that comes next. |
| **Autocomplete** | A tool that suggests the next word. Your phone keyboard has one. |

- An LLM does the same as your phone keyboard, but much better.
- Its **only job**: guess the **next word**. Then add it and guess again.

```
i drink sweet  →  tea
i drink sweet tea  →  .
```

Every layer today is a better way to make this **one guess**.

**✏️ Quick Check:** On your phone you type *"good"*. Which word will it suggest? How does it know?

**Answer:** *morning* (or *luck*). The phone has seen *"good morning"* many times, so it guesses the word that came next most often. **We do exactly this in Layer 1.**

## Layer 1: Count and guess

🪜 **You are here: Layer 1 of 7** (▲ built on Layer 0)

| New word | Meaning |
|---|---|
| **Training text** | The text the model learns from. Ours is 4 small sentences. |
| **Counter** | A Python tool that counts how many times each item appears. |

**Idea:** look at lots of text. After the word "sweet", which word came **most often**? Guess that one.

*Everyday example:* a shopkeeper notices that after "sweet", customers ask for tea most of the time. So he starts the tea.

In [36]:
text = ["i drink sweet tea",
        "we drink sweet tea",
        "i eat sweet apple",
        "we eat sweet apple"]

after_sweet = Counter()                  # a counter: word → how many times

for sentence in text:
    w = sentence.split()                 # cut into words
    for i in range(len(w) - 1):          # look at each word and the one after it
        if w[i] == "sweet":
            after_sweet[w[i + 1]] += 1   # count the word that came after "sweet"

print(after_sweet)                       # tea 2, apple 2

Counter({'tea': 2, 'apple': 2})


**It is a tie.** A **tie** means two answers have the same score, so the model cannot choose. (tea 2, apple 2)

**What breaks:** one word is not enough. We need to look at **more words**. ▲ Layer 2

**✏️ Quick Check:** Count the words that come after **"drink"**. Change `"sweet"` in the code above.

In [37]:
# your code here

**Answer:**

In [38]:
after_drink = Counter()
for sentence in text:
    w = sentence.split()
    for i in range(len(w) - 1):
        if w[i] == "drink":
            after_drink[w[i + 1]] += 1
print(after_drink)    # sweet 2: no tie here, "drink" is always followed by "sweet"

Counter({'sweet': 2})


## Layer 2: Count with more words

🪜 **You are here: Layer 2 of 7** (▲ built on Layer 1)

| New word | Meaning |
|---|---|
| **Context** | The words around (before) the word we want to guess. It gives clues. |
| **Dictionary (`{}`)** | A Python table: a **key** (what we look up) and a **value** (the answer). |

**Idea:** do not use only the last word. Use the **last two words**.

- `drink sweet` → tea
- `eat sweet` → apple

*Everyday example:* "sweet" alone is not enough. "drink sweet" and "eat sweet" are different questions. The word before it is the clue.

In [39]:
table = {}                                    # (word before, last word) → counter of next words
for sentence in text:
    w = sentence.split()
    for i in range(1, len(w) - 1):
        key = (w[i - 1], w[i])
        table.setdefault(key, Counter())[w[i + 1]] += 1

print(table[("drink", "sweet")])              # tea 2  → no tie now!
print(table[("eat", "sweet")])                # apple 2

Counter({'tea': 2})
Counter({'apple': 2})


The tie is gone. More context fixed it.

**What breaks:** try a phrase the table has **never seen**. "sugary" means almost the same as "sweet":

In [40]:
print(table.get(("drink", "sugary")))         # None: the table has no answer

None


- The table only knows **exact phrases it has seen**. It does not know "sugary" ≈ "sweet".
- To remember every phrase, the table would grow without end.

**What we need:** a way to say *"sugary" and "sweet" mean almost the same thing*. For that, **words must become numbers**. ▲ Layer 3

**✏️ Quick Check:** What does `table` say for the phrase `("i", "drink")`?

In [41]:
# your code here

**Answer:** after *i drink* comes *sweet*.

In [42]:
print(table[("i", "drink")])    # sweet 1

Counter({'sweet': 1})


## Layer 3: Words become numbers

🪜 **You are here: Layer 3 of 7** (▲ built on Layer 2)

A computer cannot read words. We do **3 small steps**:

1. **Tokens**: cut the text into pieces.
2. **Token IDs**: give each piece a number (like a roll number).
3. **Embeddings**: give each piece a **list of numbers that describes its meaning**.

| New word | Meaning |
|---|---|
| **Token** | One piece of text. A word, or part of a word. |
| **Tokenizer** | The tool that cuts text into tokens. |
| **Vocabulary** | The list of all tokens the model knows. |
| **Token ID** | The position of a token in the vocabulary (its roll number). |
| **Vector** | A list of numbers, like `[0, 2, 0, 0]`. |
| **Embedding** | The vector that describes one token's meaning. |

### 3a: Tokens

- Our toy: **1 word = 1 token** (cut at the spaces).

In [43]:
sentence = "i drink sweet"
tokens = sentence.split()        # cut at the spaces
print(tokens)                    # ['i', 'drink', 'sweet']

['i', 'drink', 'sweet']


**A real tokenizer** (the one used by GPT-4) cuts words into **pieces**.
If this cell gives an error, run `pip install tiktoken` once (it needs internet the first time).

In [44]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")          # the GPT-4 tokenizer
for text_ in ["I drink sweet tea.", "chaiwala", "unbelievable", "Lahore"]:
    ids_ = enc.encode(text_)                         # text → token IDs
    pieces = [enc.decode([i]) for i in ids_]         # each ID → its piece of text
    print(f"{text_!r:24} → {pieces}  ({len(ids_)} tokens)")

'I drink sweet tea.'     → ['I', ' drink', ' sweet', ' tea', '.']  (5 tokens)
'chaiwala'               → ['chai', 'w', 'ala']  (3 tokens)
'unbelievable'           → ['un', 'belie', 'vable']  (3 tokens)
'Lahore'                 → ['L', 'ah', 'ore']  (3 tokens)


**✏️ Quick Check:** Use `enc.encode` to count the tokens in **"Sara"** and in **"Karachi"**.

In [45]:
# your code here

**Answer:**

In [46]:
for word in ["Sara", "Karachi"]:
    ids_ = enc.encode(word)
    print(word, "→", [enc.decode([i]) for i in ids_], len(ids_), "tokens")

Sara → ['S', 'ara'] 2 tokens
Karachi → ['Kar', 'achi'] 2 tokens


### 3b: Token IDs

- The **vocabulary** is the list of all tokens the model knows.
- The **ID** of a token = its position in that list (counting starts at 0).
- Our vocabulary has 6 words. A real one has about 100,000.

In [47]:
vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]    # position 0, 1, 2, 3, 4, 5

ids = [vocab.index(t) for t in tokens]    # find each token's position
print(ids)                                # [0, 1, 3]

[0, 1, 3]


**✏️ Quick Check:** What are the IDs of **"i eat sweet"**? Guess first, then check with code.

In [48]:
# your code here

**Answer:**

In [49]:
print([vocab.index(t) for t in "i eat sweet".split()])   # [0, 2, 3]

[0, 2, 3]


**Think:** what happens with `vocab.index("mango")`? Try it. Real tokenizers never get this error. Why? *(They cut unknown words into smaller pieces they do know, like "chaiwala" → chai + w + ala.)*

### 3c: Embeddings (vectors)

- An **ID is only a label.** ID 4 is not "closer" to ID 3 in meaning.
- So each token gets a **vector**: a list of numbers that says what the word is about.
- Our vectors have 4 numbers with names: **person, drink, food, sweet**.
- `E` is a table: **one row per word**. Row 4 is the vector of "tea".
- Real models: 768 or more numbers per token, **learned** during training, with no names. We wrote ours so we can read them.

In [50]:
#             person drink food sweet
E = np.array([[3,     0,    0,   0],    # 0 i
              [0,     2,    0,   0],    # 1 drink
              [0,     0,    2,   0],    # 2 eat
              [0,     1,    1,   1],    # 3 sweet   (drink? food? not sure!)
              [0,     2,    0,   1],    # 4 tea
              [0,     0,    2,   1]],   # 5 apple
             dtype=float)

vectors = E[ids]           # take the rows for our IDs [0, 1, 3]
print(vectors)

[[3. 0. 0. 0.]
 [0. 2. 0. 0.]
 [0. 1. 1. 1.]]


**✏️ Quick Check:** Print the vector of **apple** using its ID.

In [51]:
# your code here

**Answer:**

In [52]:
print(E[vocab.index("apple")])     # [0, 0, 2, 1]: food, sweet

[0. 0. 2. 1.]


### 3d: Dot product = similarity

How do we measure if two words are close in meaning?

| New word | Meaning |
|---|---|
| **Dot product** | **Multiply the matching numbers, then add.** |
| **Similarity** | How close two meanings are. Big dot product = similar. 0 = nothing in common. |

*Example:* `tea · drink` = (0×0) + (2×2) + (0×0) + (1×0) = **4**

It is the same "multiply and add" as a CNN filter and a Dense neuron.

In [79]:
print(vocab.index("tea"))
tea = E[vocab.index("tea")]
for word in vocab:
    print(f"tea · {word:7} = {np.dot(tea, E[vocab.index(word)])}")

4
tea · i       = 0.0
tea · drink   = 4.0
tea · eat     = 0.0
tea · sweet   = 3.0
tea · tea     = 5.0
tea · apple   = 1.0


**✏️ Quick Check:** Which word is **most similar to apple** (not counting apple itself)? Use `np.dot`.

In [54]:
# your code here

**Answer:**

In [55]:
apple = E[vocab.index("apple")]
for word in vocab:
    print(f"apple · {word:7} = {np.dot(apple, E[vocab.index(word)])}")
# eat: 4  → apple is most similar to "eat"

apple · i       = 0.0
apple · drink   = 0.0
apple · eat     = 4.0
apple · sweet   = 3.0
apple · tea     = 1.0
apple · apple   = 5.0


**Link to Layer 2:** because meaning is now numbers, "sugary" would get a vector with a big **sweet** number, close to "sweet". The model no longer needs to have seen the exact phrase.

**What breaks:** look at the last word "sweet" alone, now with numbers. ▲ Layer 4

## Layer 4: Attention

🪜 **You are here: Layer 4 of 7** (▲ built on Layer 3)

| New word | Meaning |
|---|---|
| **Attention** | Each word looks at the other words and **listens more to the useful ones**. |
| **Weights** (attention weights) | How much to listen to each word. They add up to 1. |
| **Softmax** | Turns scores into weights (or chances) that add up to 1. |
| **Weighted average** | Mix the vectors, giving the bigger weights a bigger share. |

### The problem (the same tie as Layer 1)

Use only the last word, "sweet", to score every word in the vocabulary:

In [56]:
sweet = E[vocab.index("sweet")]
for word, score in zip(vocab, E @ sweet):     # E @ sweet = dot product of sweet with every row
    print(f"{word:7} {score}")
# tea 3, apple 3, sweet 3 → a tie! "sweet" needs to look back at "drink".

i       0.0
drink   2.0
eat     2.0
sweet   3.0
tea     3.0
apple   3.0


### The fix: attention

*Everyday example:* in a noisy room you hear many voices, but you listen **more** to the person talking to you.

3 small steps:

1. **Score:** dot product of "sweet" with each word in the sentence. A big score means "useful".
2. **Softmax:** scores → weights that add up to 1.
3. **Weighted average:** multiply each word's vector by its weight, then add. The result is a new "sweet" that **carries the clue from "drink"**.

> **This is the heart of the Transformer.** In the picture, the purple *Attention* box does exactly these 3 steps.

In [57]:
query = vectors[-1]                  # the last word: "sweet"

att_scores = vectors @ query         # 1. sweet · i, sweet · drink, sweet · sweet
print("scores: ", att_scores)        # [0, 2, 3]

weights = softmax(att_scores)        # 2. softmax
print("weights:", weights)           # [0.035, 0.259, 0.705]

mix = weights @ vectors              # 3. weighted average of the 3 vectors
print("sweet before:", query)        # [0, 1, 1, 1]
print("sweet after: ", mix)          # [0.105, 1.224, 0.705, 0.705]: drink went up!

scores:  [0. 2. 3.]
weights: [0.035 0.259 0.705]
sweet before: [0. 1. 1. 1.]
sweet after:  [0.105 1.224 0.705 0.705]


Look at "sweet": the **drink** number went **up** (1 → 1.22) and the **food** number went **down** (1 → 0.71). The word "sweet" now knows it is about a **drink**.

**✏️ Quick Check:** Do the weights add up to 1? Print `weights.sum()`. Which word does "sweet" listen to the least?

In [58]:
# your code here

**Answer:**

In [59]:
print(round(weights.sum(), 3))     # 1.0 (the computer may show 0.9999999: tiny rounding)
# "i": only 3.5%. The word "i" tells us nothing about tea or apple.

1.0


## Layer 5: Score every word, softmax, pick

🪜 **You are here: Layer 5 of 7** (▲ built on Layer 4)

| New word | Meaning |
|---|---|
| **Score** | A number that says how well a word fits as the next word. |
| **Chance** | A score turned into a percentage by softmax. All chances add up to 100%. |

We have a better "sweet". Now compare it with **every word in the vocabulary** (dot product again), turn the scores into chances (softmax), and pick the biggest.

In [60]:
scores = E @ mix                     # dot product of the mixed "sweet" with every word
chances = softmax(scores)

for i in np.argsort(-chances):       # biggest chance first
    print(f"{vocab[i]:7} score {scores[i]:.2f}   chance {chances[i]:.1%}")

print("Next token:", vocab[chances.argmax()])    # tea

tea     score 3.15   chance 37.4%
sweet   score 2.64   chance 22.2%
drink   score 2.45   chance 18.5%
apple   score 2.12   chance 13.2%
eat     score 1.41   chance 6.5%
i       score 0.32   chance 2.2%
Next token: tea


**The tie is gone: tea wins (37%), apple only 13%.** Attention fixed what counting could not.

**Why is "sweet" in 2nd place?** Our toy is tiny and uses the same vectors to read and to guess. A real model has a separate, learned output layer and has read enough text to know that "sweet sweet" is rare.

**✏️ Quick Check:** Change the sentence to **"i eat sweet"** and run Layers 4–5 again. Which word wins? *(Try it before the Home Problem!)*

In [61]:
# your code here

**Answer:**

In [62]:
v = E[[vocab.index(t) for t in "i eat sweet".split()]]
m = softmax(v @ v[-1]) @ v
c = softmax(E @ m)
print(vocab[c.argmax()], f"{c.max():.1%}")    # apple 37.4%

apple 37.4%


## Layer 6: Put it together, and repeat

🪜 **You are here: Layer 6 of 7** (▲ built on Layer 5)

| New word | Meaning |
|---|---|
| **Function** | A saved set of steps we can run again with one name. |
| **Loop (repeat)** | Add the new word to the text and guess again. |

One function = Layers 3 to 5. Then we **add the new token and run again**. This is how an LLM writes a whole answer, one word at a time.

In [63]:
def next_token(words):
    vectors = E[[vocab.index(w) for w in words]]      # tokens → IDs → vectors
    weights = softmax(vectors @ vectors[-1])          # attention weights for the last word
    mix = weights @ vectors                           # weighted average
    chances = softmax(E @ mix)                        # score every word → chances
    return vocab[chances.argmax()], chances


words = ["i", "drink", "sweet"]
for step in range(3):                                 # 3 passes = 3 new tokens
    word, chances = next_token(words)
    print(" ".join(words), "→", word, f"({chances.max():.0%})")
    words.append(word)                                # add the guess, then repeat

i drink sweet → tea (37%)
i drink sweet tea → tea (59%)
i drink sweet tea tea → tea (62%)


**✏️ Quick Check:** The toy says "tea tea tea". Why would a real LLM not do this?

**Answer:** Our toy has only 6 words and was never trained. A real LLM learned from huge amounts of text that "tea tea tea" almost never happens, so it gives other tokens a bigger chance. (Real chatbots also sometimes pick a token that is not the top one.)

## Layer 7 (top): The real LLM

🪜 **You are here: Layer 7 of 7** (▲ built on Layer 6)

| New word | Meaning |
|---|---|
| **Feed-forward** | A small network (like your Day 19 Dense layer) that "thinks" about each word after attention. |
| **Pre-training** | Learning from huge text by guessing hidden words. |
| **Loss** | A number that says how wrong the guess was (cross entropy, Day 22). |
| **Gradient descent** | Change the weights a little so the loss gets smaller. |
| **Fine-tuning** | Extra training on good question-and-answer examples. |
| **Inference** | Using the finished model. The weights are fixed. |

Our toy and a real LLM follow **the same recipe**. Look at the picture again:

![Transformer, annotated](Visual%20Explanation/Day_23_Transformer_Annotated.png)

| In our toy | In a real LLM |
|---|---|
| 6 words | about 100,000 tokens |
| 4 numbers per word, **we wrote them** | 768 to 12,000+ numbers per word, **learned** |
| **1** attention step | attention + feed-forward, **repeated 12 to 100+ times** (the purple block) |
| nothing learned | trained on huge text (below) |

### How is it trained? The same loop as Day 19–22

1. **Pre-training:** show a sentence with the last word hidden (`i drink sweet ___`). The model guesses. The **loss** says how wrong it was. **Gradient descent** changes the weights a little. Repeat billions of times on books, websites and code.
2. **Fine-tuning:** train more on good question-and-answer examples (and human ratings), so it behaves like a helpful assistant.
3. **Inference:** the weights are now fixed. It only does the steps we built today: guess, add, repeat.

In [64]:
print("Our toy has", E.size, "learned numbers")   # 6 words × 4 numbers = 24
# Day 19 network: 147   |   Day 20 network: 5,130   |   a big LLM: 100,000,000,000+

Our toy has 24 learned numbers


**✏️ Quick Check:** A real model has 100,000 tokens and 768 numbers per token. How many numbers are in its embedding table?

In [65]:
# your code here

**Answer:** and this is only the embedding table. The attention and feed-forward blocks add many more.

In [66]:
print(100_000 * 768)    # 76,800,000: 76.8 million, and that is only the first table

76800000


## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [81]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

# Layer 3: tokens → IDs → embeddings (person, drink, food, sweet)
vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]
tokens = "i drink sweet".split()
ids = [vocab.index(t) for t in tokens]

E = np.array([[3, 0, 0, 0], [0, 2, 0, 0], [0, 0, 2, 0],
              [0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 1]], dtype=float)
vectors = E[ids]

print(vectors)

# Layer 4: attention for the last word
weights = softmax(vectors @ vectors[-1])
mix = weights @ vectors

# Layer 5: score every word, softmax, pick
chances = softmax(E @ mix)
print("Next token:", vocab[chances.argmax()], f"({chances.max():.1%})")   # tea (37.4%)

[[3. 0. 0. 0.]
 [0. 2. 0. 0.]
 [0. 1. 1. 1.]]
Next token: tea (37.4%)


### Key takeaways

- An **LLM** = **L**arge **L**anguage **M**odel: a very big autocomplete. One job: guess the next token, then repeat.
- **Counting** (Layers 1–2) breaks: ties, and it only knows phrases it has seen.
- **Words → numbers** (tokens, IDs, embeddings): similar words get similar numbers.
- **Attention** fixes the tie: each word listens more to the useful words. It is the heart of the **Transformer**.
- **Softmax** turns scores into chances (same as Day 21). Training uses **cross entropy** (same as Day 22).
- A real LLM = the same block **stacked many times**, with billions of learned numbers. An **SLM** is the same idea, just smaller.

---
## 🎁 Bonus (optional): the same example with a real embedding model

Our table `E` has 4 numbers per word, and **we wrote them**. Now we ask a **real embedding model** for the numbers of the same 6 words.

- You will learn how an API works tomorrow (Day 24). Today, just run the cells.
- You need internet and a free key from **openrouter.ai**. No key? Skip this part. Nothing else needs it.

In [90]:
import os
from getpass import getpass
from openai import OpenAI

key = os.environ.get("OPENROUTER_API_KEY") or getpass("OpenRouter key (hidden): ")    # hidden, never write a key in code
emb_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

resp = emb_client.embeddings.create(model="nvidia/nemotron-3-embed-1b:free", input=vocab)   # our 6 words
E_real = np.array([item.embedding for item in resp.data])      # one row per word, like E

print(E.shape, "→", E_real.shape)       # (6, 4) → (6, N): same 6 words, many more numbers

AuthenticationError: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}

**Where do these numbers come from?**

- This embedding model was **trained already** by NVIDIA, on a huge amount of text (web pages, books, questions and answers). We are **not training** it. We only **use** it.
- We sent our 6 words, **each alone**. It sent back one list of numbers for each word. The model does **not** know our sentence "i drink sweet".
- *Everyday example:* a person who has read millions of pages. You ask "what is tea?" and he answers with a long list of scores. That list is the embedding.
- Our own code (attention, scores, softmax) comes **after**. That part is a tiny toy with nothing learned.

In [74]:
print(E, "→\n", E_real)       # (6, 4) → (6, N): same 6 words, many more numbers

[[3. 0. 0. 0.]
 [0. 2. 0. 0.]
 [0. 0. 2. 0.]
 [0. 1. 1. 1.]
 [0. 2. 0. 1.]
 [0. 0. 2. 1.]] →
 [[-0.02  -0.154 -0.016 ... -0.036 -0.027  0.011]
 [ 0.014 -0.026  0.014 ... -0.039 -0.031 -0.016]
 [ 0.019 -0.04   0.04  ... -0.015 -0.036 -0.008]
 [ 0.011 -0.058  0.006 ... -0.027 -0.004  0.01 ]
 [-0.002 -0.104 -0.027 ...  0.038  0.004 -0.001]
 [-0.004 -0.096  0.01  ...  0.01  -0.01   0.006]]


**Compare:** how close is "tea" to each word? We use **cosine similarity** (the dot product, after scaling both lists to the same length). Close to 1 = similar meaning.

In [75]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

t = vocab.index("tea")
print("word      ours  real")
for i, word in enumerate(vocab):
    print(f"{word:7} {cosine(E[t], E[i]):6.2f} {cosine(E_real[t], E_real[i]):5.2f}")

word      ours  real
i         0.00  0.23
drink     0.89  0.36
eat       0.00  0.32
sweet     0.77  0.25
tea       1.00  1.00
apple     0.20  0.25


Now run **the same steps** (attention, scores, softmax) with the real numbers. Real vectors are small, so we multiply the scores by 10 to let softmax see the differences.

In [83]:
En = E_real / np.linalg.norm(E_real, axis=1, keepdims=True)    # scale every row to length 1


def next_word_real(words, scale=10):
    v = En[[vocab.index(w) for w in words]]               # tokens → IDs → real vectors
    weights = softmax(scale * (v @ v[-1]))                # attention weights for the last word
    mix = weights @ v                                     # weighted average
    chances = softmax(scale * (En @ mix))                 # score every word → chances
    return weights, chances


weights, chances = next_word_real(["i", "drink", "sweet"])
print("weights:", weights)
for i in np.argsort(-chances):
    print(f"{vocab[i]:7} {chances[i]:.1%}")

weights: [0.001 0.001 0.999]
sweet   99.6%
eat     0.1%
drink   0.1%
i       0.1%
apple   0.1%
tea     0.1%


Look at the result. Does it still say **tea**? Probably **not**. The complete code below shows why.

- Our toy has **no learned attention**. A real LLM learns it, and has a learned output layer too.
- What matters: the **steps are the same**: embeddings → attention → scores → softmax. Only the numbers changed.

**✏️ Quick Check:** Run `next_word_real` for **"i eat sweet"**. Print the top word.

In [71]:
# your code here

**Answer:**

In [77]:
weights, chances = next_word_real(["i", "eat", "sweet"])
print(vocab[chances.argmax()])     # your answer may differ from our toy (apple)

sweet


## ✅ Complete code (Bonus)

All the bonus steps in one cell. You can run this cell alone. It also prints the **attention weights**, so we can see what happens.

In [ ]:
import os
import numpy as np
from getpass import getpass
from openai import OpenAI


def softmax(scores):
    e = np.exp(scores - scores.max())
    return e / e.sum()


vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]

# 1. real embeddings of our 6 words (the model was trained by NVIDIA; we only use it)
key = os.environ.get("OPENROUTER_API_KEY") or getpass("OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
resp = client.embeddings.create(model="nvidia/nemotron-3-embed-1b:free", input=vocab)
En = np.array([item.embedding for item in resp.data])
En = En / np.linalg.norm(En, axis=1, keepdims=True)       # scale every row to length 1
print("shape:", En.shape)                                 # (6, 2048)


# 2. the same steps as the Class Problem: attention → scores → softmax
def next_word_real(words, scale=10):
    v = En[[vocab.index(w) for w in words]]               # tokens → IDs → real vectors
    weights = softmax(scale * (v @ v[-1]))                # attention weights for the last word
    mix = weights @ v                                     # weighted average
    chances = softmax(scale * (En @ mix))                 # score every word → chances
    return weights, chances


for sentence in (["i", "drink", "sweet"], ["i", "eat", "sweet"]):
    weights, chances = next_word_real(sentence)
    print(" ".join(sentence), "→", vocab[chances.argmax()], f"({chances.max():.0%})")
    print("   attention weights:", dict(zip(sentence, weights.round(2))))

### Why is it not "tea"?

Look at the **attention weights** above. Almost all of the weight is on **"sweet"** (the last word), so the mix is just "sweet" again. And the word closest to "sweet" is **"sweet"** itself.

- **It is not the vocabulary.** Any 6 words would do the same.
- A word is always most similar to **itself**. In real embeddings this beats the other words.
- NVIDIA's model learned **what words mean**. It did **not** learn to guess the next word.
- Our hand-made table `E` worked because **we designed the numbers** for this one sentence.
- A real LLM **learns** its attention and its output layer in training. Our toy has nothing learned.

**So:** real embeddings are great for **meaning** (similarity, search). Guessing the next word needs a **trained LLM**. That is what we use tomorrow (Day 24).

---
# Problem: "i eat sweet ___"

Do the **same layers** for the new sentence.

1. Tokens and IDs (Layer 3).
2. Vectors (`E[ids]`).
3. Attention: scores, weights, mix (Layer 4). Compare "sweet" before and after: which number went up?
4. Score every word, softmax (Layer 5). What is the next token, and its chance?
5. In 2 lines: only one word changed. Which layer made the answer change, and how?

In [ ]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

# Layer 3: tokens → IDs → embeddings (person, drink, food, sweet)
vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]
tokens = "i drink sweet".split()
ids = [vocab.index(t) for t in tokens]

E = np.array([[3, 0, 0, 0], [0, 2, 0, 0], [0, 0, 2, 0],
              [0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 1]], dtype=float)
vectors = E[ids]

print(ids)

# Layer 4: attention for the last word
weights = softmax(vectors @ vectors[-1])
mix = weights @ vectors

# Layer 5: score every word, softmax, pick
chances = softmax(E @ mix)
print("Next token:", vocab[chances.argmax()], f"({chances.max():.1%})")   # tea (37.4%)

[0, 2, 3]
Next token: apple (37.4%)


In [85]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

# Layer 3: tokens → IDs → embeddings (person, drink, food, sweet)
vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]
tokens = "i drink sweet".split()
ids = [vocab.index(t) for t in tokens]

# E = np.array([[3, 0, 0, 0], [0, 2, 0, 0], [0, 0, 2, 0],
#               [0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 1]], dtype=float)
# 
vectors = E_real[ids]

print(vectors)

# Layer 4: attention for the last word
weights = softmax(vectors @ vectors[-1])
mix = weights @ vectors

# Layer 5: score every word, softmax, pick
chances = softmax(E_real @ mix)
print("Next token:", vocab[chances.argmax()], f"({chances.max():.1%})")   # tea (37.4%)

[[-0.02  -0.154 -0.016 ... -0.036 -0.027  0.011]
 [ 0.014 -0.026  0.014 ... -0.039 -0.031 -0.016]
 [ 0.011 -0.058  0.006 ... -0.027 -0.004  0.01 ]]
Next token: sweet (21.0%)
